<div style="width: 50%;">
    <img src="../ONS_Logo_Digital_Colour_Landscape_English_RGB.svg" alt="ONS Logo">
</div>

# ClassifAI Demo

This demo uses a mock occupations dataset to show how ClassifAI matches unlabelled job descriptions to SOC codes using an existing knowledgebase.

In [ ]:
import re
import pandas as pd

from classifai.indexers import VectorStore
from classifai.indexers.dataclasses import VectorStoreSearchInput
from classifai.indexers.hooks import (
    CapitalisationStandardisingHook,
    DeduplicationHook,
)
from classifai.indexers.hooks.hook_factory import HookBase

from demo_utils import (
    KNOWLEDGEBASE_PATH,
    load_uncoded_input,
    load_vectoriser,
    make_query_input,
    prepare_knowledgebase,
)

prepare_knowledgebase()
uncoded_input = load_uncoded_input()
vectoriser = load_vectoriser()


basic_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
)

---
### Example Responses

The examples below use a small batch of uncoded occupation descriptions. In a real workflow, these could come from survey responses, form submissions, or operational data waiting to be coded.

In this demo, we have the following `uncoded_input` which is a batch of free text responses we want to match to SOC codes.

In [ ]:
uncoded_input[["id", "query"]]

---
### 1. Return A Best Match

At the simplest level, you pass in uncoded text: `query_text` 

and get back the highest-ranked suggestion for each response: `doc_label`, `doc_text`.

In [ ]:
basic_vectorstore.search(query=uncoded_input, n_results=1)

---
### 2. Return A Shortlist For Review

When a single suggestion is not enough, the same search can return a ranked shortlist for review.

In [ ]:
basic_vectorstore.search(query=uncoded_input, n_results=3)

---
### 3. Return Extra Context With Each Match

The matched knowledgebase `doc_label`'s and `doc_text` is useful on its own, but sometimes extra linked information is desireable.

Extra columns such as `sector` in this example, can support analyst review, help group similar results, or allow downstream postprocessing.

In [ ]:
metadata_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    meta_data={"sector": str},  # we specify what extra context we want here.
)

metadata_vectorstore.search(query=uncoded_input, n_results=1)

---
### 4. Use `search_preprocess` Hooks To Handle Messy Input

Hooks let you adapt behaviour without rewriting the search call.

Here the input is deliberately inconsistent. `CapitalisationStandardisingHook` normalises the text before embedding. 
>This is useful as data from different sources may having varying cases, leading to slight differences in matching.

In [ ]:
messy_input = make_query_input([
    "TOMATO FARMER",
    "Machine Learning ENGINEER",
    "pHd StUdEnT",
])

hook_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={
        "search_preprocess": CapitalisationStandardisingHook(method="lower")
    },  # we add any hooks here.
)


basic_results = basic_vectorstore.search(query=messy_input, n_results=1)

capitalisation_hook_results = hook_vectorstore.search(query=messy_input, n_results=1)

print("Basic results")
display(basic_results)

print("After Capitalisation Hook")
display(capitalisation_hook_results)

---
### 5. Use `search_postprocess` hooks to clean up the shortlist.

If several knowledgebase entries share the same label, the raw shortlist can contain repeated labels. 

`DeduplicationHook` trims that down to one best match per label
> This is useful as it reduces the number of matches a coder will have to look through to assign a code.

In [ ]:
dedup_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={
        "search_postprocess": DeduplicationHook(score_aggregation_method="max")
    },  # we add any hooks here.
)

review_query = make_query_input([
    "Tomato Farmer: Cultivates and harvests tomatoes in large greenhouse operations."
])

basic_results = basic_vectorstore.search(query=review_query, n_results=5)
clean_results = dedup_vectorstore.search(query=review_query, n_results=5)

print("Basic results")
display(basic_results)

print("After deduplication")
display(clean_results)

---
### 6. Add a Custom Domain-Specific Hook

Built-in hooks cover common cases, but users often have their own domain specific problems. Custom hooks can solve these sorts of problems in a concise and reusable way.

> For instance if our domain had a lot of shorthand abbriviations that are not commonly known, we could write a hook to make them long form.

In [ ]:
class AbbreviationHook(HookBase):
    def __init__(self, colname: str = "query"):
        super().__init__(colname=colname, hook_type="pre_processing")
        self.colname = colname

    def _substitute_abbreviations(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r"\bml\b", "machine learning", text)
        text = re.sub(r"\bdev\b", "developer", text)
        text = re.sub(r"\bons\b", "office for national statistics", text)
        return text

    def __call__(self, input_data: VectorStoreSearchInput) -> VectorStoreSearchInput:
        processed_input = input_data.copy()
        processed_input[self.colname] = (
            processed_input[self.colname]
            .astype(str)
            .apply(self._substitute_abbreviations)
        )
        return input_data.__class__.validate(processed_input)


custom_abreviations_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={"search_preprocess": AbbreviationHook()},  # we add any hooks here.
)

custom_input = make_query_input(["dev", "ml engineer", "ons employee"])

basic_results = basic_vectorstore.search(query=custom_input, n_results=1)

custom_abreviations_results = custom_abreviations_vectorstore.search(
    query=custom_input, n_results=1
)

print("Basic results")
display(basic_results)

print("After custom abreviation hook")
display(custom_abreviations_results)

---
### 6. Add a thresholding hook

Sometimes the best match is still not a good enough match. A thresholding hook lets us set the minimum score we are willing to accept, so low-confidence suggestions could be ignored.

The hook runs after the search and removes results below the threshold. The rest of the search stays exactly the same.

In [ ]:
class ThresholdingHook(HookBase):
    def __init__(self, threshold: float = 0.8):
        super().__init__(colname="score", hook_type="post_processing")
        self.threshold = threshold

    def __call__(self, input_data):
        thresholded_input = input_data[input_data["score"] >= self.threshold]
        return input_data.__class__.validate(thresholded_input)


thresholded_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={
        "search_postprocess": ThresholdingHook(threshold=0.8)
    },  # we add any hooks here.
)

basic_results = basic_vectorstore.search(query=review_query, n_results=5)
custom_threshold_results = thresholded_vectorstore.search(
    query=review_query,
    n_results=5,
)

print("Basic results")
display(basic_results)

print("After custom threshold hook")
display(custom_threshold_results)

---
Thanks for looking at our demo
have a look at https://github.com/datasciencecampus/classifai for more info!